<a href="https://colab.research.google.com/github/manamendraJN/SE4050-Fraud-Detection/blob/CNN-1D/notebooks/CNN1D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SE4050 Deep Learning Assignment
CNN-1D — Credit Card Fraud Detection

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, copy

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, f1_score, precision_score, recall_score, accuracy_score,
    auc as auc_fn
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


## 2. Mount Google Drive and load preprocessed splits

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_DIR = "/content/drive/MyDrive/SE4050-Fraud-Detection"
DATA_DIR    = f"{PROJECT_DIR}/data/processed"
RESULTS_DIR = f"{PROJECT_DIR}/results/cnn1d"
CONFIGS_DIR = f"{PROJECT_DIR}/configs"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CONFIGS_DIR, exist_ok=True)

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df   = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name} fraud rate: {split['Class'].mean()*100:.3f}%")

Train: (198608, 31) Val: (42559, 31) Test: (42559, 31)
  train fraud rate: 0.167%
  val fraud rate: 0.167%
  test fraud rate: 0.167%


In [4]:
TARGET_COL = "Class"
feature_cols = [c for c in train_df.columns if c != TARGET_COL]
N_FEATURES = len(feature_cols)
print(f"Using {N_FEATURES} features:", feature_cols)

def to_xy(df):
    X = df[feature_cols].values.astype(np.float32)
    y = df[TARGET_COL].values.astype(np.float32)
    return X, y

X_train, y_train = to_xy(train_df)
X_val,   y_val   = to_xy(val_df)
X_test,  y_test  = to_xy(test_df)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

BATCH_SIZE = 2048
def make_loader(X, y, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False)

train_loader = make_loader(X_train, y_train, shuffle=True)

Using 30 features: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_log']
Shapes: (198608, 30) (42559, 30) (42559, 30)


## 3. Prepare features/target for CNN-1D

In [5]:
TARGET_COL = "Class"
feature_cols = [c for c in train_df.columns if c != TARGET_COL]
N_FEATURES = len(feature_cols)
print(f"Using {N_FEATURES} features:", feature_cols)

def to_xy(df):
    X = df[feature_cols].values.astype(np.float32)
    y = df[TARGET_COL].values.astype(np.float32)   # float for BCEWithLogitsLoss
    return X, y

X_train, y_train = to_xy(train_df)
X_val,   y_val   = to_xy(val_df)
X_test,  y_test  = to_xy(test_df)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

BATCH_SIZE = 2048
def make_loader(X, y, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False)

train_loader = make_loader(X_train, y_train, shuffle=True)

Using 30 features: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_log']
Shapes: (198608, 30) (42559, 30) (42559, 30)
